# Pennylane Fundamentals

## Quantum Operations

In [1]:
import pennylane as qml
from pennylane import numpy as np

### Quantum State Preparation

In [2]:
dev = qml.device("default.qubit", wires = 3)

@qml.qnode(dev)
def prep_circuit(alpha, beta, gamma):
    """
    Prepares the state alpha|001> + beta|010> + gamma|100>.
    Args:
    alpha, beta, gamma (np.complex): The coefficients of the quantum state
    to prepare.
    Returns:
    (np.array): The quantum state
    """

    ####################
    ###YOUR CODE HERE###
    ####################
    state = [0, alpha, beta, 0, gamma, 0, 0, 0]
    
    # Normalisation au cas où
    state = state / np.linalg.norm(state)
    
    qml.StatePrep(state, wires=[0, 1, 2])
    
    return qml.state()

alpha, beta, gamma = 1/np.sqrt(3), 1/np.sqrt(3), 1/np.sqrt(3),

print("The prepared state is", prep_circuit(alpha, beta, gamma))


The prepared state is [0.        +0.j 0.57735027+0.j 0.57735027+0.j 0.        +0.j
 0.57735027+0.j 0.        +0.j 0.        +0.j 0.        +0.j]


### A circuit  with single-qubit gates

In [3]:
dev = qml.device("default.qubit", wires = 2)

@qml.qnode(dev)
def single_qubit_gates(theta, phi):
    """
    Implements the quantum circuit shown in the statement
    Args:
    - theta, phi (float): The arguments for the RX and RZ gates, respectively
    Returns:
    - (np.array): The output quantum state.
    
    """
    qml.Hadamard(0)
    qml.Hadamard(1)
    qml.T(0)
    qml.S(1)
    qml.RX(theta, 0)
    qml.RZ(phi, 1)

    ####################
    ###YOUR CODE HERE###
    ####################
    
    return qml.state()

theta, phi = np.pi/3, np.pi/4
print("The output state of the circuit is: ", single_qubit_gates(theta, phi))

The output state of the circuit is:  [ 0.49572243-0.39667667j -0.07003593+0.63102146j  0.30438071-0.0652631j
 -0.16908169+0.26137765j]


### A circuit  with multi-qubit gates

In [4]:
dev = qml.device("default.qubit", wires = 3)

@qml.qnode(dev)
def multi_qubit_gates(theta,phi):
    """
    Applies the circuit shown the figure above
    Args:
    theta, phi (float): parameters of the CRX and CRY gates, in that order.
    Returns:
    - (np.array): the quantum state
    """
    
    #####################
    ###YOUR CODE HERE####
    #####################
    qml.Hadamard(0)
    qml.CRY(phi, wires = [0,1])
    qml.CRX(theta, wires = [1,2])
    qml.S(1)
    qml.T(2)
    qml.Toffoli(wires = [0,1,2])
    qml.SWAP(wires = [0,2])

    return qml.state()

theta, phi = np.pi/3, np.pi/4
print("The output state is: \n", multi_qubit_gates(theta, phi))

The output state is: 
 [0.70710678+0.j         0.65328148+0.j         0.        +0.j
 0.09567086+0.09567086j 0.        +0.j         0.        +0.j
 0.        +0.j         0.        +0.23434479j]


### Gates under control

In [5]:
dev = qml.device("default.qubit", wires = 3)

@qml.qnode(dev)
def ctrl_circuit(theta,phi):
    """Implements the circuit shown in the Codercise statement
    Args:
        theta (float): Rotation angle for RX
        phi (float): Rotation angle for RY
    Returns:
        (numpy.array): The output state of the QNode
    """

    ####################
    ###YOUR CODE HERE###
    ####################
    qml.RY(phi, 0)
    qml.H(1)
    qml.RX(theta, 2)

    qml.ctrl(qml.S(wires=1), control=0)
    qml.ctrl(qml.T(2), control = 1, control_values = 0)
    qml.ctrl(qml.Hadamard(wires=0), control=2)

    return qml.state()

### Kicking U back

In [6]:
dev = qml.device("default.qubit", wires = 2)

@qml.qnode(dev)
def phase_kickback(matrix):
    """Applies phase kickback to a single-qubit operator whose matrix is known
    Args:S
    - matrix (numpy.array): A 2x2 matrix
    Returns:
    - (numpy.array): The output state after applying phase kickback
    """

    ####################
    ###YOUR CODE HERE###
    ####################

    qml.Hadamard(0)
    qml.ControlledQubitUnitary(matrix,  wires = [0,1])
    qml.Hadamard(0)

    return qml.state()

matrix = np.array([[-0.69165024-0.50339329j,  0.28335369-0.43350413j],
    [ 0.1525734 -0.4949106j , -0.82910055-0.2106588j ]])

print("The state after phase kickback is: \n" , phase_kickback(matrix))

The state after phase kickback is: 
 [ 0.15417488-0.25169664j  0.0762867 -0.2474553j   0.84582512+0.25169664j
 -0.0762867 +0.2474553j ]


### Do, apply, Undo

In [7]:
dev = qml.device("default.qubit", wires = 3)

def do(k):

    qml.StatePrep([1,k], wires = [0], normalize = True)

def apply(theta):

    qml.IsingXX(theta, wires = [1,2])

@qml.qnode(dev)
def do_apply_undo(k,theta):
    """
    Applies V, controlled-U, and the inverse of V
    Args: 
    - k, theta: The parameters for do and apply (V and U) respectively
    Returns:
    - (np.array): The output state of the routine
    """

    ####################
    ###YOUR CODE HERE###
    ####################
    do(k)
    qml.ctrl(apply, control = 0)(theta)
    qml.adjoint(do)(k)

    return qml.state()

k, theta = 0.5, 0.8

print("The output state is: \n", do_apply_undo(k, theta))

The output state is: 
 [ 0.9842122+0.j          0.       +0.j          0.       +0.j
  0.       -0.07788367j -0.0315756+0.j          0.       +0.j
  0.       +0.j          0.       -0.15576734j]
